# Instacart Data Pipeline
## Stage 4: Analytics — Business Questions
**Source schema:** `workspace.instacart_gold` | **Output schema:** `workspace.instacart_analytics`

### What this notebook does

Answers the 3 business questions from the assignment, plus a 4th question the
team picked, reading only from Gold. Each answer is materialized as a small
table under a new `instacart_analytics` schema — the dashboard should query
these, not `gold_fact_order_product` directly, since that fact table is
~33.8M rows and re-aggregating it on every dashboard load is slow.

### Business questions covered

1. Which products and departments are purchased most frequently?
2. How does customer purchasing behavior change by day of week and hour of day?
3. Which products have the highest reorder behavior?
4. **Team's question:** Which products are most often bought together?

### Design notes

- Every query reads from `gold_fact_order_product` / `gold_dim_product` /
  `gold_dim_order` only — never Bronze or Silver, consistent with the rest of
  this pipeline.
- Q3 (reorder rate) requires a minimum order-line count before a product
  qualifies, so a product bought once (100% or 0% reorder rate on n=1) can't
  rank above genuinely popular ones.
- Q4 (bought-together) restricts the self-join to the 200 most-purchased
  products before joining. A full self-join across all 33.8M fact rows finds
  mostly noise anyway — rare pairings aren't useful for a cross-sell
  dashboard — and the top-200 filter cuts the join size by roughly two
  orders of magnitude while still catching every pairing worth acting on.
- A short validation pass runs at the end, in the same PASS/REVIEW pattern
  as the Silver and Gold notebooks.

## Part 0: Setup

In [0]:
%sql
-- Owner: Maeve
-- Name: 00 - Analytics Setup
-- Purpose: Create the analytics schema that holds each business question's materialized answer table.
-- Grain: N/A - schema creation only.

USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS instacart_analytics;
USE SCHEMA instacart_analytics;

## Part 1: Build

### Question 1: Which products and departments are purchased most frequently?

Two cuts of the same question — department-level for the high-level read,
product-level for the specific SKUs. Both are pure volume counts off the
fact table, joined to `gold_dim_product` for the names.

In [0]:
%sql
-- Owner: Maeve
-- Name: 01 - Analytics Top Departments
-- Purpose: Rank departments by order-line volume to answer "which departments are purchased most frequently".
-- Grain: One row per department.

CREATE OR REPLACE TABLE analytics_top_departments AS
SELECT
    p.department_name,
    COUNT(*) AS order_line_count,
    COUNT(DISTINCT f.order_id) AS distinct_orders
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_product p ON f.product_id = p.product_id
GROUP BY p.department_name
ORDER BY order_line_count DESC;

SELECT * FROM analytics_top_departments;

In [0]:
%sql
-- Owner: Maeve
-- Name: 02 - Analytics Top Products
-- Purpose: Rank individual products by how often they're purchased.
-- Grain: One row per product, top 50 by order-line volume.

CREATE OR REPLACE TABLE analytics_top_products AS
SELECT
    p.product_name,
    p.department_name,
    p.aisle_name,
    COUNT(*) AS order_line_count
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name, p.department_name, p.aisle_name
ORDER BY order_line_count DESC
LIMIT 50;

SELECT * FROM analytics_top_products;

### Question 2: How does customer purchasing behavior change by day of week and hour of day?

Uses `order_day_name`, already computed in `gold_dim_order`, joined onto the
fact table. Two tables: order-line volume by day + hour (the main answer —
this is the one to chart as a heatmap), and average basket size by day as a
companion metric (not asked for directly, but a natural staffing/inventory
read on the same cut).

In [0]:
%sql
-- Owner: Maeve
-- Name: 03 - Analytics Day Hour Patterns
-- Purpose: Aggregate order-line volume by day of week and hour of day to show when customers shop.
-- Grain: One row per (order_day_name, order_hour_of_day).

CREATE OR REPLACE TABLE analytics_day_hour_patterns AS
SELECT
    o.order_day_name,
    f.order_hour_of_day,
    COUNT(*) AS order_line_count,
    COUNT(DISTINCT f.order_id) AS distinct_orders
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_order o ON f.order_id = o.order_id
GROUP BY o.order_day_name, f.order_hour_of_day
ORDER BY order_line_count DESC;

SELECT * FROM analytics_day_hour_patterns;

-- In the result grid above, switch to the "Visualization" tab and pick a
-- heatmap (day on one axis, hour on the other, order_line_count as the
-- value) -- that's the clearest way to present this answer.

In [0]:
%sql
-- Owner: Maeve
-- Name: 04 - Analytics Basket Size By Day
-- Purpose: Average items per order by day of week - a companion metric to the day/hour volume pattern above.
-- Grain: One row per order_day_name.

CREATE OR REPLACE TABLE analytics_basket_size_by_day AS
SELECT
    o.order_day_name,
    COUNT(*) / COUNT(DISTINCT f.order_id) AS avg_items_per_order
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_order o ON f.order_id = o.order_id
GROUP BY o.order_day_name
ORDER BY avg_items_per_order DESC;

SELECT * FROM analytics_basket_size_by_day;

### Question 3: Which products have the highest reorder behavior?

Reorder rate = share of a product's order-lines where `reordered = true`.
Ranking on raw reorder rate with no volume floor lets a product bought once
(rate = 100% or 0% on n=1) outrank genuinely popular products, so this
requires at least 500 order-lines before a product qualifies.

In [0]:
%sql
-- Owner: Maeve
-- Name: 05 - Analytics Reorder Rates
-- Purpose: Rank products by reorder rate, restricted to products with enough volume for the rate to be meaningful.
-- Grain: One row per product, minimum 500 order-lines, top 50 by reorder rate.

CREATE OR REPLACE TABLE analytics_reorder_rates AS
SELECT
    p.product_name,
    p.department_name,
    COUNT(*) AS total_order_lines,
    SUM(CASE WHEN f.reordered THEN 1 ELSE 0 END) AS reorder_count,
    ROUND(AVG(CASE WHEN f.reordered THEN 1.0 ELSE 0.0 END), 4) AS reorder_rate
FROM workspace.instacart_gold.gold_fact_order_product f
JOIN workspace.instacart_gold.gold_dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name, p.department_name
HAVING COUNT(*) >= 500
ORDER BY reorder_rate DESC
LIMIT 50;

SELECT * FROM analytics_reorder_rates;

### Question 4 (team's question): Which products are most often bought together?

This is what the Instacart dataset was originally built for (it's literally
the "Market Basket Analysis" dataset), so it gives the dashboard
a cross-sell angle the other three don't cover.

`filtered_fact` restricts the self-join to the 200 most-purchased products
first, `f1.product_id < f2.product_id` keeps each pair once instead of twice
(A-B and B-A) and drops a product pairing with itself.

In [0]:
%sql
-- Owner: Maeve
-- Name: 06 - Analytics Product Pairs
-- Purpose: Find product pairs most frequently purchased in the same order, restricted to the
--          top 200 products by volume so the self-join stays tractable at fact-table scale.
-- Grain: One row per unordered product pair (product_a, product_b), top 50 by co-purchase count.

CREATE OR REPLACE TABLE analytics_product_pairs AS
WITH top_products AS (
    SELECT product_id
    FROM workspace.instacart_gold.gold_fact_order_product
    GROUP BY product_id
    ORDER BY COUNT(*) DESC
    LIMIT 200
),
filtered_fact AS (
    SELECT f.order_id, f.product_id
    FROM workspace.instacart_gold.gold_fact_order_product f
    JOIN top_products t ON f.product_id = t.product_id
)
SELECT
    p1.product_name AS product_a,
    p2.product_name AS product_b,
    COUNT(*) AS times_bought_together
FROM filtered_fact f1
JOIN filtered_fact f2
    ON f1.order_id = f2.order_id
    AND f1.product_id < f2.product_id
JOIN workspace.instacart_gold.gold_dim_product p1 ON f1.product_id = p1.product_id
JOIN workspace.instacart_gold.gold_dim_product p2 ON f2.product_id = p2.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY times_bought_together DESC
LIMIT 50;

SELECT * FROM analytics_product_pairs;

## Part 2: Validate

Lightweight sanity pass over the 6 analytics tables built above — same
PASS/REVIEW pattern as the Silver and Gold validation notebooks: every
table should have rows, and `reorder_rate` should never fall outside
0-1.

In [0]:
%sql
-- Owner: Maeve
-- Name: 07 - Analytics Validation
-- Purpose: Confirm every analytics table built above has rows and that reorder_rate stays within a valid 0-1 range.
-- Grain: One validation summary row per analytics table.

WITH validation AS (

    SELECT 'analytics_top_departments' AS table_name, COUNT(*) AS row_count,
           0 AS out_of_range_values
    FROM analytics_top_departments

    UNION ALL

    SELECT 'analytics_top_products', COUNT(*), 0
    FROM analytics_top_products

    UNION ALL

    SELECT 'analytics_day_hour_patterns', COUNT(*), 0
    FROM analytics_day_hour_patterns

    UNION ALL

    SELECT 'analytics_basket_size_by_day', COUNT(*), 0
    FROM analytics_basket_size_by_day

    UNION ALL

    SELECT 'analytics_reorder_rates', COUNT(*),
           SUM(CASE WHEN reorder_rate < 0 OR reorder_rate > 1 THEN 1 ELSE 0 END)
    FROM analytics_reorder_rates

    UNION ALL

    SELECT 'analytics_product_pairs', COUNT(*), 0
    FROM analytics_product_pairs

)
SELECT
    *,
    CASE WHEN row_count > 0 AND out_of_range_values = 0 THEN 'PASS' ELSE 'REVIEW' END AS status
FROM validation
ORDER BY table_name;

## Analytics Layer: Summary

| Table | Answers | Notes |
|---|---|---|
| `analytics_top_departments` | Q1 — departments | |
| `analytics_top_products` | Q1 — products | top 50 |
| `analytics_day_hour_patterns` | Q2 | pair with a heatmap visualization |
| `analytics_basket_size_by_day` | Q2 (companion metric) | avg items/order by day |
| `analytics_reorder_rates` | Q3 | min. 500 order-lines to qualify |
| `analytics_product_pairs` | Q4 — team's question | top-200-product self-join, top 50 pairs |

**Expected result:** `status = 'PASS'` on all 6 rows in the validation pass.

**Next stage:** Dashboard — read from `workspace.instacart_analytics`, not
`gold_fact_order_product` directly, so the dashboard stays fast regardless
of fact-table size.